# LOSO con Modelos Fundacionales de Series Temporales

Este notebook implementa y explica paso a paso la clasificación de crisis epilépticas usando **Chronos2** (Amazon) y **Moirai** (Salesforce).

## Idea central: forecasting como detector de anomalías

Los modelos fundacionales de series temporales se entrenan sobre millones de series de dominios muy diversos. Eso les permite capturar patrones temporales generales: periodicidad, tendencias, ruido estacionario, etc.

**Un EEG interictal (sin crisis) es relativamente predecible**: el cerebro genera oscilaciones rítmicas (alfa, beta, theta) que un modelo bien entrenado puede anticipar con error bajo.

**Un EEG ictal (crisis activa) es impredecible**: las descargas epileptiformes rompen abruptamente los patrones habituales. El modelo falla al predecir → error alto.

La cadena completa es:
```
ventana EEG  →  context (800 muestras)  →  modelo  →  forecast (200 muestras)
                                                              ↓
                             MAE(forecast, señal real)  =  anomaly score
                                                              ↓
                                score ≥ threshold  →  Seizure = 1
```

El threshold se elige **sin ver el paciente de test**: se optimiza sobre los pacientes restantes (LOSO).

## 1. Imports

In [ ]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, auc, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
)

## 2. Parámetros de configuración

Todos los hiperparámetros en un único bloque para facilitar la experimentación.

### ¿Por qué estos valores?

| Parámetro | Valor | Razón |
|-----------|-------|-------|
| `CHANNEL` | `"EEG F3"` | Canal frontal, buena relación señal/ruido y sensible a crisis generalizadas |
| `WINDOW_SIZE` | `1000` | 10 s a 100 Hz — ventana clásica en detección de crisis EEG |
| `WINDOW_OVERLAP` | `0.25` | 25% solapamiento → más ventanas sin coste de etiquetado extra |
| `CONTEXT_LENGTH` | `800` | 8 s de contexto — suficiente para que el modelo aprenda el ritmo basal del paciente en esa ventana |
| `PREDICTION_LENGTH` | `200` | 2 s de horizonte — corto para que el EEG normal sea predecible, largo para que el ictal diverja |

> **Nota**: `CONTEXT_LENGTH + PREDICTION_LENGTH = WINDOW_SIZE`. Las 1000 muestras de cada ventana se dividen: 800 al modelo como historia conocida, 200 como horizonte a predecir.

In [ ]:
# --- Datos ---
RAW_DIR        = Path("data/raw/csv-data")
OUTPUT_DIR     = Path("images/results/loso")
CHANNEL        = "EEG F3"
WINDOW_SIZE    = 1000   # muestras por ventana (10 s a 100 Hz)
WINDOW_OVERLAP = 0.25   # fracción de solapamiento entre ventanas

# --- Modelos ---
CONTEXT_LENGTH    = 800   # muestras de contexto que recibe el modelo
PREDICTION_LENGTH = 200   # muestras que el modelo debe predecir
DEVICE            = "cuda"  # usar 'cpu' si no hay GPU
BATCH_SIZE        = 32

# --- IDs de modelos en HuggingFace / repositorio ---
CHRONOS_MODEL_ID = "amazon/chronos-2"
MOIRAI_MODEL_ID  = "Salesforce/moirai-2.0-R-small"

# --- LOSO ---
SEED       = 42
MAX_VAL_WINDOWS = 0  # 0 = usar todos los pacientes restantes para calibrar el threshold

## 3. Carga y ventaneado de datos

Los CSVs crudos están en `data/raw/csv-data/PN_XX/*.csv_clipped.csv`. Cada fichero contiene **toda una sesión** de un paciente con columnas:
- `idPatient`, `idSession` — identificadores
- `EEG Fp1`, `EEG F3`, ... — señal EEG en µV (o equivalente)
- `Seizure` — etiqueta 0/1 por muestra

El **windowing** parte cada sesión en segmentos de `WINDOW_SIZE` muestras con `WINDOW_OVERLAP` de solapamiento. Cada segmento recibe un `window_id` único. La etiqueta de la ventana es `max(Seizure)` dentro de ella: si hay aunque sea un sample ictal, la ventana entera es Seizure=1.

> Usamos `max` en vez de `any` o `mean` porque el modelo debe alertar aunque la crisis ocupe solo parte de la ventana.

In [ ]:
_RENAME_COLS = {"EEG CZ": "EEG Cz", "EEG FP2": "EEG Fp2"}  # normalizar variantes de nombre


def _window_df(df: pd.DataFrame, window_size: int, overlap: float) -> pd.DataFrame:
    """Ventana deslizante por sesión. Devuelve DataFrame con columna window_id."""
    step = int(window_size * (1 - overlap))
    windows = []
    for session_id in df["idSession"].unique():
        s_df = df[df["idSession"] == session_id].reset_index(drop=True)
        for start in range(0, len(s_df) - window_size + 1, step):
            w = s_df.iloc[start:start + window_size].copy()
            w["window_id"] = f"{session_id}_{start}"
            windows.append(w)
    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()


def load_patient(patient_id: str) -> pd.DataFrame:
    """Lee todos los CSVs clipped de un paciente, aplica windowing y devuelve el DF."""
    patient_dir = RAW_DIR / patient_id
    dfs = []
    for csv_file in sorted(patient_dir.glob("*_clipped.csv")):
        try:
            d = pd.read_csv(csv_file)
            if len(d) > 0:
                dfs.append(d)
        except Exception as e:
            print(f"  [WARN] {csv_file.name}: {e}")

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    rename = {k: v for k, v in _RENAME_COLS.items() if k in df.columns}
    if rename:
        df.rename(columns=rename, inplace=True)
    df.fillna(0, inplace=True)
    return _window_df(df, WINDOW_SIZE, WINDOW_OVERLAP)


def list_patients() -> List[str]:
    return sorted(d.name for d in RAW_DIR.iterdir() if d.is_dir())


def load_all_patients() -> pd.DataFrame:
    parts = []
    for pid in list_patients():
        windowed = load_patient(pid)
        if len(windowed) > 0:
            parts.append(windowed)
            print(f"  [{pid}] {len(windowed):,} filas ventaneadas")
    df = pd.concat(parts, ignore_index=True)
    print(f"\nTotal: {len(df):,} filas — {df['idPatient'].nunique()} pacientes")
    return df


print("Cargando datos...")
all_df = load_all_patients()

## 4. De DataFrame a `series_dict`

Los modelos trabajan con **diccionarios `{window_id → array de floats}`**. Necesitamos convertir el DataFrame ventaneado a este formato.

Para cada ventana:
1. Extraemos los valores del canal seleccionado → array de `CONTEXT_LENGTH + PREDICTION_LENGTH` muestras
2. Si la ventana tiene menos muestras de lo necesario → la descartamos
3. La etiqueta es `max(Seizure)` → 1 si alguna muestra de la ventana es ictal

El resultado son dos dicts paralelos:
- `series`: `{wid: array(1000,)}`
- `labels`: `{wid: 0 | 1}`

In [ ]:
def build_series_and_labels(
    df: pd.DataFrame,
    channel: str = CHANNEL,
    max_windows: int = 0,
) -> Tuple[Dict[str, np.ndarray], Dict[str, int]]:
    """Extrae series y etiquetas de un DataFrame ventaneado para un canal dado.

    Descarta ventanas con menos de CONTEXT_LENGTH + PREDICTION_LENGTH muestras.
    """
    min_len = CONTEXT_LENGTH + PREDICTION_LENGTH  # = WINDOW_SIZE si no se recorta
    grouped = df.groupby("window_id", sort=False)
    window_ids = list(grouped.groups.keys())

    if max_windows > 0:
        window_ids = window_ids[:max_windows]

    series: Dict[str, np.ndarray] = {}
    labels: Dict[str, int] = {}

    for wid in window_ids:
        g = grouped.get_group(wid)
        values = g[channel].to_numpy(dtype=np.float32)
        if len(values) < min_len:
            continue
        series[str(wid)] = values[:min_len]
        labels[str(wid)] = int(g["Seizure"].max())

    n_sz = sum(v for v in labels.values())
    print(f"  {len(series)} ventanas — seizure={n_sz}, non-seizure={len(series) - n_sz}")
    return series, labels

## 5. Chronos2 — configuración

### ¿Qué es Chronos2?

**Chronos2** (Amazon, 2024) es un modelo fundacional de series temporales basado en T5. Se entrena tokenizando valores reales en bins cuantizados y aplicando predicción de secuencias. Devuelve distribuciones probabilísticas sobre el futuro.

Lo usamos en modo **zero-shot**: no lo afinamos, simplemente le pasamos el contexto EEG y le pedimos que prediga las siguientes 200 muestras. Sólo usamos la mediana (quantile 0.5) de su distribución.

### ¿Por qué `predict_df`?

`Chronos2Pipeline.predict_df()` acepta un DataFrame en formato **long** con columnas `id`, `timestamp`, `target`. Esto permite procesar múltiples series en un solo batch de manera eficiente. El modelo procesa todas las ventanas juntas → un forward pass por batch de 32 ventanas.

### ¿Por qué `"amazon/chronos-2"` y no la variante `small` o `large`?

La variante base ofrece un compromiso entre capacidad y velocidad de inferencia. Las variantes `small` son más rápidas pero capturan menos matices; `large` es más potente pero impracticable en el bucle LOSO con decenas de pacientes.

In [ ]:
from chronos import Chronos2Pipeline

print(f"Cargando Chronos2 desde '{CHRONOS_MODEL_ID}' en {DEVICE}...")
# device_map='cuda' mueve automáticamente los pesos al primer dispositivo GPU disponible.
# Usar 'cpu' si no hay GPU (mucho más lento).
chronos_pipeline = Chronos2Pipeline.from_pretrained(
    CHRONOS_MODEL_ID,
    device_map=DEVICE,
)
print("Chronos2 listo.")

### Formato de entrada para Chronos2

`predict_df` espera un DataFrame **long** con tres columnas:

| columna | tipo | descripción |
|---------|------|-------------|
| `id` | str | identificador de la serie (= window_id) |
| `timestamp` | int | paso temporal relativo (0, 1, 2, ..., CONTEXT_LENGTH-1) |
| `target` | float | valor de la señal EEG en µV |

Usamos timestamps **enteros relativos** (no fechas absolutas) porque Chronos2 no usa información temporal absoluta para EEG — solo la forma de la señal importa.

La salida es también un DataFrame long con columna `"0.5"` (mediana) y una fila por timestep de predicción por serie.

In [ ]:
def chronos_forecast(
    series_dict: Dict[str, np.ndarray],
) -> Dict[str, np.ndarray]:
    """Predice las siguientes PREDICTION_LENGTH muestras para cada ventana.

    Devuelve dict {window_id -> array(PREDICTION_LENGTH,)} con la mediana del forecast.
    """
    # Construir DataFrame long con el contexto de cada ventana
    rows = []
    for sid, values in series_dict.items():
        context = values[:CONTEXT_LENGTH]  # primeras 800 muestras
        for t, v in enumerate(context):
            rows.append({"id": sid, "timestamp": t, "target": float(v)})
    context_df = pd.DataFrame(rows)

    # Inferencia: Chronos2 recibe el contexto y devuelve el forecast
    pred_df = chronos_pipeline.predict_df(
        context_df,
        prediction_length=PREDICTION_LENGTH,
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        quantile_levels=[0.5],  # solo necesitamos la mediana para el MAE
    )

    # Parsear salida: agrupar por serie y ordenar por timestamp
    preds: Dict[str, np.ndarray] = {}
    pred_col = "0.5"  # columna de la mediana en la salida de predict_df
    for sid, g in pred_df.groupby("id", sort=False):
        pred_values = g.sort_values("timestamp")[pred_col].to_numpy(dtype=np.float32)
        preds[str(sid)] = pred_values[:PREDICTION_LENGTH]

    return preds


print("Función chronos_forecast definida.")

## 6. Moirai — configuración

### ¿Qué es Moirai?

**Moirai** (Salesforce, 2024) es otro modelo fundacional de series temporales, basado en una arquitectura transformer propia entrenada sobre el corpus **LOTSA** (Large-scale Open Time Series Archive). A diferencia de Chronos, trabaja directamente con valores reales usando patch embeddings.

Usamos la variante `moirai-2.0-R-small`: la "R" indica que es una versión refinada con mayor calidad de predicción.

### ¿Por qué usar ambos modelos?

Chronos y Moirai tienen **arquitecturas y datos de entrenamiento diferentes** → errores no correlacionados → comparar sus scores permite entender si el patrón de anomalía es robusto o depende del modelo.

### Estructura de carga de Moirai

Moirai separa el **módulo** (pesos del modelo) de la **clase de predicción** (`Moirai2Forecast`). El módulo se carga una vez; el predictor se instancia por fold porque necesita los metadatos del dataset (dimensiones, longitudes).

In [ ]:
# Importamos los componentes de uni2ts (librería oficial de Moirai)
from uni2ts.model.moirai import Moirai2Forecast, Moirai2Module

# GluonTS es el framework de series temporales que usa Moirai como interfaz de datos
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split as gluonts_split

print(f"Cargando módulo Moirai desde '{MOIRAI_MODEL_ID}'...")
# Moirai2Module contiene los pesos del transformer.
# Se carga UNA VEZ y se reutiliza en todos los folds — es zero-shot, no hay fine-tuning.
moirai_module = Moirai2Module.from_pretrained(MOIRAI_MODEL_ID)
print("Módulo Moirai listo.")

### Formato de entrada para Moirai

Moirai usa **GluonTS** como interfaz de datos. El flujo es:

1. **DataFrame ancho** (`wide_df`): columnas = series IDs, filas = timesteps, index = DatetimeIndex ficticio. Las fechas no importan para EEG — solo sirven como índice para GluonTS.

2. **`PandasDataset`**: convierte el DataFrame ancho al formato interno de GluonTS.

3. **`split`**: divide cada serie en contexto (pasado) y horizonte (futuro). El parámetro `offset=-PREDICTION_LENGTH` significa "las últimas PREDICTION_LENGTH muestras son el futuro".

4. **`generate_instances`**: genera instancias de predicción con una ventana de contexto de `CONTEXT_LENGTH` muestras.

5. **`Moirai2Forecast`**: envuelve el módulo en una clase de predicción que incluye el predictor batched.

> El predictor se crea dentro de la función porque `Moirai2Forecast` necesita conocer las dimensiones del dataset (`feat_dynamic_real_dim`, etc.) que solo se conocen al crear el `PandasDataset`.

In [ ]:
def moirai_forecast(
    series_dict: Dict[str, np.ndarray],
) -> Dict[str, np.ndarray]:
    """Predice las siguientes PREDICTION_LENGTH muestras para cada ventana con Moirai.

    Devuelve dict {window_id -> array(PREDICTION_LENGTH,)} con la mediana del forecast.
    """
    ordered_ids = list(series_dict.keys())

    # Paso 1: DataFrame ancho — cada columna es una serie (ventana EEG)
    # El índice DatetimeIndex es obligatorio para GluonTS aunque las fechas son ficticias.
    # freq="10ms" equivale a 100 Hz (1 muestra cada 10 ms).
    wide_df = pd.DataFrame({sid: series_dict[sid] for sid in ordered_ids})
    wide_df.index = pd.date_range(start="2000-01-01", periods=len(wide_df), freq="10ms")

    # Paso 2: Convertir a formato GluonTS
    ds = PandasDataset(dict(wide_df))

    # Paso 3: Split — las últimas PREDICTION_LENGTH muestras son el horizonte
    _, test_template = gluonts_split(ds, offset=-PREDICTION_LENGTH)

    # Paso 4: Generar instancias de predicción
    # windows=1: una sola predicción por serie (no rolling forecast)
    # distance=PREDICTION_LENGTH: el step entre ventanas (irrelevante con windows=1)
    test_data = test_template.generate_instances(
        prediction_length=PREDICTION_LENGTH,
        windows=1,
        distance=PREDICTION_LENGTH,
    )

    # Paso 5: Construir el predictor
    # Moirai2Forecast necesita saber el número de features adicionales del dataset
    # (covariables dinámicas). En EEG no hay covariables → 0.
    model = Moirai2Forecast(
        module=moirai_module,
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        target_dim=1,  # serie univariante
        feat_dynamic_real_dim=getattr(ds, "num_feat_dynamic_real", 0),
        past_feat_dynamic_real_dim=getattr(ds, "num_past_feat_dynamic_real", 0),
    )
    predictor = model.create_predictor(batch_size=BATCH_SIZE)

    # Paso 6: Inferencia
    inputs = list(test_data.input)
    forecasts = list(predictor.predict(inputs))

    preds: Dict[str, np.ndarray] = {}
    for idx, forecast in enumerate(forecasts):
        if idx >= len(ordered_ids):
            break
        sid = ordered_ids[idx]
        # quantile(0.5) devuelve la mediana del forecast distribucional
        pred_values = np.asarray(forecast.quantile(0.5), dtype=np.float32)
        preds[sid] = pred_values[:PREDICTION_LENGTH]

    return preds


print("Función moirai_forecast definida.")

## 7. De forecast a score: MAE como anomaly score

### ¿Por qué MAE?

El **Mean Absolute Error** entre el forecast y la señal real es una métrica simple e interpretable: cuánto se equivoca el modelo en µV de media.

- **EEG normal** → el modelo predice bien → MAE bajo
- **EEG ictal** → el modelo falla → MAE alto

Alternativas descartadas:
- **MSE**: penaliza cuadráticamente los outliers, más sensible a artefactos que a crisis
- **Negative log-likelihood**: requiere modelar la distribución completa del EEG
- **RMSE**: equivalente a MAE en términos de ranking para el threshold

### Calibración del threshold

El score continuo (MAE) se convierte en binario con un **threshold**. Para elegirlo:
1. Barremos 201 candidatos (percentiles de la distribución de scores)
2. Para cada candidato computamos F1-macro sobre los pacientes del conjunto de validación
3. Elegimos el threshold que maximiza F1-macro

F1-macro en vez de F1 binario porque el dataset es desbalanceado (muchas más ventanas non-seizure que seizure). F1-macro trata ambas clases por igual.

In [ ]:
def compute_scores(
    series_dict: Dict[str, np.ndarray],
    labels: Dict[str, int],
    preds: Dict[str, np.ndarray],
) -> Tuple[List[str], np.ndarray, np.ndarray]:
    """Calcula el MAE entre forecast y señal real para cada ventana.

    Devuelve (window_ids, y_true, scores) como arrays alineados.
    """
    window_ids, y_true, scores = [], [], []

    for sid, pred in preds.items():
        if sid not in series_dict:
            continue
        values = series_dict[sid]
        # El futuro real son las muestras [CONTEXT_LENGTH, CONTEXT_LENGTH+PREDICTION_LENGTH)
        actual_future = values[CONTEXT_LENGTH:CONTEXT_LENGTH + PREDICTION_LENGTH]
        mae = float(np.mean(np.abs(actual_future - pred[:PREDICTION_LENGTH])))

        window_ids.append(sid)
        y_true.append(labels[sid])
        scores.append(mae)

    return window_ids, np.array(y_true, dtype=np.int64), np.array(scores, dtype=np.float32)


def tune_threshold(scores: np.ndarray, y_true: np.ndarray) -> float:
    """Encuentra el threshold que maximiza F1-macro sobre el conjunto dado.

    Barre 201 percentiles de la distribución de scores para encontrar el óptimo.
    """
    if np.unique(y_true).size < 2:
        # Solo hay una clase → usar la mediana como fallback
        thr = float(np.median(scores))
        print(f"  [WARN] Una sola clase en validación. Threshold=mediana={thr:.4f}")
        return thr

    candidates = np.unique(np.quantile(scores, np.linspace(0.0, 1.0, 201)))
    best_thr, best_f1 = float(candidates[0]), -1.0

    for thr in candidates:
        y_pred = (scores >= thr).astype(np.int64)
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)

    print(f"  Threshold: {best_thr:.4f}  (F1-macro val={best_f1:.4f})")
    return best_thr


def evaluate(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> Dict:
    """Clasifica con el threshold dado y devuelve métricas + predicciones binarias."""
    y_pred = (scores >= threshold).astype(np.int64)
    metrics = {
        "Accuracy":  float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "F1 Score":  float(f1_score(y_true, y_pred, zero_division=0)),
        "F1 Macro":  float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "F1 Micro":  float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
    }
    if np.unique(y_true).size > 1:
        try:
            metrics["ROC AUC"] = float(roc_auc_score(y_true, scores))
        except Exception:
            metrics["ROC AUC"] = None
    else:
        metrics["ROC AUC"] = None
    return metrics, y_pred


print("Funciones de scoring y evaluación definidas.")

## 8. LOSO — bucle principal

### ¿Qué es LOSO?

**Leave-One-Subject-Out**: para cada paciente `P_i`:
- **Test** = todas las ventanas de `P_i` (el modelo nunca las ha visto)
- **Validación** = todas las ventanas de los demás pacientes `{P_j : j ≠ i}`

El threshold se calibra en validación y se aplica en test. Al final tenemos una predicción para cada paciente sin contaminación de información.

### Zero-shot: el modelo se carga una sola vez

Chronos2 y Moirai son zero-shot: los pesos no cambian entre folds. Se cargan una vez antes del bucle y se reutilizan. Esto es crucial porque cargar un LLM cuesta varios segundos.

```
para cada paciente P_i:
    val_series,  val_labels  = ventanas de todos los demás
    test_series, test_labels = ventanas de P_i

    val_preds   = modelo.forecast(val_series)     # zero-shot
    val_scores  = MAE(val_preds, val_series)
    threshold   = argmax_F1macro(val_scores, val_labels)

    test_preds  = modelo.forecast(test_series)    # zero-shot
    test_scores = MAE(test_preds, test_series)
    y_pred      = test_scores >= threshold
    métricas    = eval(y_pred, test_labels)
```

In [ ]:
def run_fold(
    patient_id: str,
    test_df: pd.DataFrame,
    val_df: pd.DataFrame,
    forecast_fn,  # chronos_forecast o moirai_forecast
) -> Optional[Dict]:
    """Ejecuta un fold LOSO para un modelo dado.

    Devuelve dict con métricas y predicciones, o None si el fold debe saltarse.
    """
    print(f"    Construyendo series de test ({patient_id})...")
    test_series, test_labels = build_series_and_labels(test_df, max_windows=0)

    print(f"    Construyendo series de validación (restantes)...")
    val_series, val_labels = build_series_and_labels(val_df, max_windows=MAX_VAL_WINDOWS)

    if not test_series:
        print(f"    [SKIP] {patient_id}: sin ventanas válidas de test")
        return None
    if not val_series:
        print(f"    [SKIP] {patient_id}: sin ventanas válidas de validación")
        return None

    # Calibración del threshold sobre validación
    print("    Forecasting validación...")
    val_preds = forecast_fn(val_series)
    _, y_val, val_scores = compute_scores(val_series, val_labels, val_preds)
    threshold = tune_threshold(val_scores, y_val)

    # Evaluación sobre el paciente de test
    print("    Forecasting test...")
    test_preds = forecast_fn(test_series)
    test_ids, y_test, test_scores = compute_scores(test_series, test_labels, test_preds)

    if len(test_ids) == 0:
        print(f"    [SKIP] {patient_id}: sin predicciones alineadas en test")
        return None

    metrics, y_pred = evaluate(y_test, test_scores, threshold)
    return {
        "patient_id": patient_id,
        "metrics":    metrics,
        "y_true":     y_test,
        "y_pred":     y_pred,
        "y_score":    test_scores,
        "window_ids": test_ids,
        "threshold":  threshold,
    }


print("Función run_fold definida.")

## 9. Ejecución del LOSO

Aquí se ejecuta el bucle completo para ambos modelos. Los modelos ya están cargados (`chronos_pipeline`, `moirai_module`) — solo pasamos la función de forecast correspondiente.

In [ ]:
MODELS = {
    "chronos2": ("Chronos2",  chronos_forecast),
    "moirai":   ("Moirai",    moirai_forecast),
}

patients = sorted(all_df["idPatient"].unique())
print(f"Pacientes ({len(patients)}): {patients}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_model_results: Dict[str, List[Dict]] = {}

for model_key, (model_name, forecast_fn) in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {model_name.upper()} — LOSO ({len(patients)} folds)")
    print("=" * 60)

    fold_results: List[Dict] = []

    for fold_idx, patient_id in enumerate(patients):
        print(f"\n  Fold {fold_idx + 1}/{len(patients)}  test={patient_id}")

        test_df = all_df[all_df["idPatient"] == patient_id]
        val_df  = all_df[all_df["idPatient"] != patient_id]

        result = run_fold(patient_id, test_df, val_df, forecast_fn)

        if result is not None:
            fold_results.append(result)
            roc = result["metrics"].get("ROC AUC")
            print(
                f"    F1={result['metrics'].get('F1 Score', 0):.4f}  "
                f"F1-macro={result['metrics'].get('F1 Macro', 0):.4f}  "
                f"ROC-AUC={'N/A' if roc is None else f'{roc:.4f}'}"
            )

    if fold_results:
        all_model_results[model_name] = fold_results
    else:
        print(f"  [WARN] No hay folds válidos para {model_name}")

## 10. Agregación de resultados y guardado

In [ ]:
def aggregate_metrics(fold_results: List[Dict]) -> Dict[str, Dict[str, float]]:
    """Media y desviación estándar de cada métrica entre folds."""
    buckets: Dict[str, List[float]] = {}
    for res in fold_results:
        for metric, value in res["metrics"].items():
            if value is not None:
                buckets.setdefault(metric, []).append(value)
    return {
        m: {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        for m, vals in buckets.items()
    }


# Resumen por modelo
all_aggregated: Dict[str, Dict] = {}
for model_name, fold_results in all_model_results.items():
    agg = aggregate_metrics(fold_results)
    all_aggregated[model_name] = agg
    print(f"\n{model_name} — LOSO ({len(fold_results)} folds)")
    print("-" * 50)
    for metric, stats in agg.items():
        print(f"  {metric:<14} {stats['mean']:.4f} ± {stats['std']:.4f}")


# Guardar CSV de métricas por fold
rows = []
for model_name, fold_results in all_model_results.items():
    for res in fold_results:
        row = {"model": model_name, "patient": res["patient_id"]}
        row.update({k: v for k, v in res["metrics"].items() if v is not None})
        rows.append(row)

csv_path = OUTPUT_DIR / "loso_fold_metrics.csv"
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"\nMétricas por fold guardadas en: {csv_path}")


# Guardar predicciones concatenadas por modelo (para plot_loso_results.py)
preds_dir = OUTPUT_DIR / "predictions"
preds_dir.mkdir(parents=True, exist_ok=True)

for model_name, fold_results in all_model_results.items():
    model_key = model_name.lower().replace(" ", "")
    pd.DataFrame({
        "y_true":  np.concatenate([r["y_true"]  for r in fold_results]),
        "y_score": np.concatenate([r["y_score"] for r in fold_results]),
    }).to_csv(preds_dir / f"{model_key}_loso_predictions.csv", index=False)

## 11. Plots

Matriz de confusión sumada (todos los folds) y curvas ROC por modelo.

In [ ]:
graphs_dir = OUTPUT_DIR / "graphs"
graphs_dir.mkdir(parents=True, exist_ok=True)


def plot_confusion(fold_results: List[Dict], model_name: str, save_path: Path) -> None:
    total_cm = np.zeros((2, 2), dtype=np.int64)
    for res in fold_results:
        total_cm += confusion_matrix(res["y_true"], res["y_pred"], labels=[0, 1])

    tn, fp, fn, tp = total_cm.ravel()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(total_cm, cmap="Blues")
    ax.set_title(f"{model_name} — LOSO ({len(fold_results)} folds)\nTN={tn} FP={fp} FN={fn} TP={tp}", fontsize=12)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Seizure", "Seizure"])
    ax.set_yticklabels(["No Seizure", "Seizure"])
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(total_cm[r, c]), ha="center", va="center", fontsize=14)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Confusion matrix: {save_path}")


def plot_roc_curves(all_results: Dict[str, List[Dict]], save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot([0, 1], [0, 1], "k--", label="Random", linewidth=1.5)
    for model_name, fold_results in all_results.items():
        y_true_all  = np.concatenate([r["y_true"]  for r in fold_results])
        y_score_all = np.concatenate([r["y_score"] for r in fold_results])
        if np.unique(y_true_all).size < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true_all, y_score_all)
        ax.plot(fpr, tpr, linewidth=2.0, label=f"{model_name} (AUC={auc(fpr, tpr):.3f})")
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC — Foundation Models (LOSO)"); ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  ROC curves: {save_path}")


for model_name, fold_results in all_model_results.items():
    model_key = model_name.lower().replace(" ", "")
    plot_confusion(fold_results, model_name, graphs_dir / f"loso_confusion_{model_key}.png")

plot_roc_curves(all_model_results, graphs_dir / "loso_roc_curves.png")

print("\nLOSO completado.")